In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_unstructured import UnstructuredLoader

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

In [8]:
vector_store = Chroma(
    collection_name="init_collection",
    embedding_function=embeddings,
    persist_directory="data/chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [13]:


# with Unstructured, we don't need to use a chunking strategy like RecursiveCharacterTextSplitter
# we can instead do semantic splitting (like group text under same title/heading together)
# docs (langchain) - https://docs.langchain.com/oss/python/integrations/document_loaders/unstructured_file#chunking
# docs (unstructured) - https://docs.unstructured.io/open-source/core-functionality/chunking
loader = UnstructuredLoader(
    file_path="data/Reinforcement Learning An Introduction (Adaptive Computation and Machine Learning series) (Sutton, Richard S., Barto, Andrew G.).pdf",
    chunking_strategy="by_title",
    max_characters=1500,
    new_after_n_chars=1000,
    overlap=100,
)

docs = loader.load()

f"len of docs : {len(docs)}"

'len of docs : 1437'

In [15]:
document_ids = vector_store.add_documents(documents=docs)

In [16]:
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs



In [ ]:
retrieve_context("what is Q learning")